In [101]:
# Step 1: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Export to DataBase
from sqlalchemy import create_engine, types
from dotenv import dotenv_values
from scipy.stats import ttest_ind, shapiro
import statsmodels.api as sm
import statsmodels.formula.api as smf
import scipy.stats as stats

In [28]:
# Load config
config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_pass = config['POSTGRES_PASS']
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
DB_URL = f"postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}"
# Now building the URL with the values from the .env file
engine = create_engine(DB_URL, echo=False)


# Hypothesis 5: A higher dietary sodium-to-potassium ratio is positively associated with an increased prevalence of hypertension(elevated systolic and diastolic blood pressure)

In [ ]:
try:
    engine = create_engine(DB_URL, echo=False) # echo=True prints SQL statements (useful for debugging)
    print("SQLAlchemy engine created successfully.")
except Exception as e:
    print(f"Error creating SQLAlchemy engine: {e}")
    print("Please check your database connection details and ensure the PostgreSQL driver (psycopg2-binary) is installed.")
  
    exit()

SQLAlchemy engine created successfully.


In [63]:
table_to_read = 'demographic_data' 

sql_query = f"SELECT * FROM {pg_schema}.{table_to_read}"

In [ ]:
try:
    df_demo = pd.read_sql_query(sql_query, con=engine)
    print(f"\nSuccessfully read table '{table_to_read}' into a DataFrame.")
    print("\n--- First 5 rows of the DataFrame ---")
    print(df.head())
    print("\n--- DataFrame Info ---")
    df.info()
except Exception as e:
    print(f"Error reading SQL table '{table_to_read}': {e}")
    print("Possible reasons: Table does not exist, incorrect table name, incorrect schema, or connection issue.")

In [ ]:
df_demo.columns

In [59]:
table_to_read = 'hypertension_risk_factor_ratios' 

sql_query = f"SELECT * FROM {pg_schema}.{table_to_read}"
df_ratios = pd.read_sql_query(sql_query, con=engine)

In [75]:
table_to_read = 'merged_demo_hypertensive_bmi_data' 

sql_query = f"SELECT * FROM {pg_schema}.{table_to_read}"
df_demo_bmi = pd.read_sql_query(sql_query, con=engine) 

In [80]:
df_demo_bmi

,Participant_ID,Age,Gender,Hypertensive,BMI,WHR,BMI_Category,Abdominal_Obesity
0,130378,43,male,True,27.0,1.0,overweight,True
1,130379,66,male,True,33.5,1.0,obese,True
2,130380,44,female,False,29.7,1.0,overweight,True
3,130386,34,male,False,30.2,1.0,obese,True
4,130387,68,female,True,42.6,0.8,obese,False
...,...,...,...,...,...,...,...,...
7796,142306,9,male,False,15.4,NaN,underweight,None
7797,142307,49,female,True,NaN,NaN,None,None
7798,142308,50,male,False,26.4,1.0,overweight,True
7799,142309,40,male,True,25.5,0.9,overweight,True


In [78]:
df_ratios = df_ratios.rename(columns={'seqn_no': 'Participant_ID'})
df_merged = pd.merge(df_demo_bmi,df_ratios,on='Participant_ID',how='inner')  # Or 'inner' if you only want participants with both data

df_analysis = df_merged[[
    'Participant_ID',
    'Age',
    'Gender',
    'Hypertensive',
    'BMI',
    'sodium_to_potassium_ratio'
]]

In [79]:

# Export to CSV

file_path = "../data/analysis_data/hypertension_demo_nutrient_analysis.csv" 

try:
    df_analysis.to_csv(file_path, index=False, encoding='utf-8')
    print(f"DataFrame successfully saved to: {file_path}")
except Exception as e:
    print(f"Error saving DataFrame to CSV: {e}")
    

DataFrame successfully saved to: ../data/analysis_data/hypertension_demo_nutrient_analysis.csv


In [85]:
df_analysis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5971 entries, 0 to 5970
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Participant_ID             5971 non-null   int64  
 1   Age                        5971 non-null   int64  
 2   Gender                     5971 non-null   object 
 3   Hypertensive               5971 non-null   bool   
 4   BMI                        5917 non-null   float64
 5   sodium_to_potassium_ratio  5971 non-null   float64
dtypes: bool(1), float64(2), int64(2), object(1)
memory usage: 239.2+ KB


In [ ]:
# --- 2. Data Preparation ---
df_analysis['Hypertensive'] = df_analysis['Hypertensive'].astype(bool)
df_analysis['sodium_to_potassium_ratio'] = pd.to_numeric(df_analysis['sodium_to_potassium_ratio'], errors='coerce')

# Drop NA if any
df_analysis.dropna(subset=['sodium_to_potassium_ratio', 'Hypertensive'], inplace=True)

# ---  Normality Check (Shapiro-Wilk test) ---

In [90]:
# --- 3. Normality Check (Shapiro-Wilk test) ---
print("\n--- Normality Test (Shapiro-Wilk) ---")
for label, group in df_analysis.groupby('Hypertensive'):
    stat, p = shapiro(group['sodium_to_potassium_ratio'])
    status = 'normal' if p > 0.05 else 'not normal'
    print(f"Hypertensive = {label}: W = {stat:.3f}, p = {p:.4f} → Data is {status}.")


--- Normality Test (Shapiro-Wilk) ---
Hypertensive = False: W = 0.004, p = 0.0000 → Data is not normal.
Hypertensive = True: W = 0.005, p = 0.0000 → Data is not normal.


Shapiro-Wilk: The result shows that the Data is not Normal

# --- T-test ---

In [93]:
# --- 4. T-test ---
ratio_h = df_analysis[df_analysis['Hypertensive']]['sodium_to_potassium_ratio']
ratio_nh = df_analysis[~df_analysis['Hypertensive']]['sodium_to_potassium_ratio']

t_stat, p_val = ttest_ind(ratio_h, ratio_nh, equal_var=False, alternative='greater')

print("\n--- T-test ---")
print(f"Hypertensive mean: {ratio_h.mean():.2f}, n={len(ratio_h)}")
print(f"Non-Hypertensive mean: {ratio_nh.mean():.2f}, n={len(ratio_nh)}")
print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_val:.4f}")


--- T-test ---
Hypertensive mean: 78177561885943562483910296982066811401116540944683390295415913043912627847168.00, n=2749
Non-Hypertensive mean: 35075443543023156982766603622848152257310993200809966993325399874328439291904.00, n=3222
T-statistic: 0.507
P-value: 0.3062


T-test: Shows if hypertensive group has significantly higher sodium-to-potassium ratios.

Regression: Adjusts for confounders like Age, Gender, and BMI. If Hypertensive is still significant here, the association is stronger.

In [107]:
# --- 5. Regression Analysis ---
print("\n--- Multiple Regression Analysis ---")


# Regression model
formula = 'sodium_to_potassium_ratio ~ Hypertensive + Age + Gender + BMI'
model = smf.ols(formula=formula, data=df_analysis).fit()

print(model.summary())



--- Multiple Regression Analysis ---
                                OLS Regression Results                               
Dep. Variable:     sodium_to_potassium_ratio   R-squared:                       0.001
Model:                                   OLS   Adj. R-squared:                  0.000
Method:                        Least Squares   F-statistic:                     1.688
Date:                       Thu, 12 Jun 2025   Prob (F-statistic):              0.150
Time:                               10:18:36   Log-Likelihood:            -1.0779e+06
No. Observations:                       5917   AIC:                         2.156e+06
Df Residuals:                           5912   BIC:                         2.156e+06
Df Model:                                  4                                         
Covariance Type:                   nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
-------------

c:\Users\susan\miniconda3\lib\site-packages\scipy\stats\_stats_py.py:1076: RuntimeWarning: overflow encountered in square
  s = s**2
c:\Users\susan\miniconda3\lib\site-packages\numpy\_core\_methods.py:127: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
